# DermAI — Binary Classifier: Psoriasis vs Not

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `dermAI_dataset.zip` to your Google Drive root
3. Run all cells in order

Expected: ~30-60 min on T4.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile, os, pathlib

ZIP_PATH = '/content/drive/MyDrive/dermAI_dataset.zip'
EXTRACT_DIR = '/content/dermAI'
os.makedirs(EXTRACT_DIR, exist_ok=True)

print('Extracting...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_DIR)

data_dir = pathlib.Path(EXTRACT_DIR) / 'prepared'
for split in ['train', 'val']:
    for cls in sorted((data_dir / split).iterdir()):
        n = len(list(cls.glob('*.jpg')))
        print(f'  {split}/{cls.name}: {n}')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import time, json
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from torchvision.models import EfficientNet_B0_Weights

DATA_DIR  = Path('/content/dermAI/prepared')
MODEL_DIR = Path('/content/drive/MyDrive/dermAI_model')
MODEL_DIR.mkdir(exist_ok=True)

EPOCHS_PHASE1 = 15
EPOCHS_PHASE2 = 15
BATCH_SIZE    = 64
LR_PHASE1     = 1e-3
LR_PHASE2     = 5e-5
IMG_SIZE      = 224

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(DATA_DIR / 'train', transform=train_tf)
val_ds   = datasets.ImageFolder(DATA_DIR / 'val',   transform=val_tf)

class_names = train_ds.classes  # ['not_psoriasis', 'psoriasis']
num_classes = len(class_names)
print(f'Classes: {class_names}')
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Class weights — handles 6:1 imbalance
class_counts = [0] * num_classes
for _, label in train_ds.samples:
    class_counts[label] += 1
total = sum(class_counts)
weights = torch.tensor([total / (num_classes * c) for c in class_counts], dtype=torch.float32)
print(f'Counts: {dict(zip(class_names, class_counts))}')
print(f'Weights: {dict(zip(class_names, [f"{w:.2f}" for w in weights.tolist()]))}')

In [ ]:
def build_model(num_classes):
    model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    for param in model.features.parameters():
        param.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 256),
        nn.SiLU(),
        nn.Dropout(p=0.2),
        nn.Linear(256, num_classes),
    )
    return model

def unfreeze_last_blocks(model, n_blocks=3):
    blocks = list(model.features.children())
    for block in blocks[-n_blocks:]:
        for param in block.parameters():
            param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Unfroze {n_blocks} blocks. Trainable: {trainable:,}')

model = build_model(num_classes).to(device)
criterion = nn.CrossEntropyLoss(weight=weights.to(device))
print('Model ready.')

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=device.type=='cuda'):
            out = model(imgs)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(labels)
        correct += (out.argmax(1) == labels).sum().item()
        total += len(labels)
    return total_loss / total, correct / total

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            loss = criterion(out, labels)
            total_loss += loss.item() * len(labels)
            correct += (out.argmax(1) == labels).sum().item()
            total += len(labels)
    return total_loss / total, correct / total

print('Helpers ready.')

In [ ]:
best_val_acc = 0.0
scaler = torch.cuda.amp.GradScaler(enabled=device.type=='cuda')

print(f'=== Phase 1: frozen backbone ({EPOCHS_PHASE1} epochs) ===')
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=LR_PHASE1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_PHASE1)

for epoch in range(1, EPOCHS_PHASE1 + 1):
    t0 = time.time()
    _, train_acc = train_epoch(model, train_loader, criterion, optimizer, scaler)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion)
    scheduler.step()
    print(f'Epoch {epoch:02d}/{EPOCHS_PHASE1} | train={train_acc:.2%} val={val_acc:.2%} | loss={val_loss:.3f} | {time.time()-t0:.0f}s')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_DIR / 'best_model.pt')
        print(f'  -> saved ({val_acc:.2%})')

print(f'Phase 1 best: {best_val_acc:.2%}')

In [ ]:
print(f'=== Phase 2: partial unfreeze ({EPOCHS_PHASE2} epochs) ===')
unfreeze_last_blocks(model, n_blocks=3)
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR_PHASE2
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_PHASE2)

for epoch in range(1, EPOCHS_PHASE2 + 1):
    t0 = time.time()
    _, train_acc = train_epoch(model, train_loader, criterion, optimizer, scaler)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion)
    scheduler.step()
    print(f'Epoch {epoch:02d}/{EPOCHS_PHASE2} | train={train_acc:.2%} val={val_acc:.2%} | loss={val_loss:.3f} | {time.time()-t0:.0f}s')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_DIR / 'best_model.pt')
        print(f'  -> saved ({val_acc:.2%})')

with open(MODEL_DIR / 'class_names.json', 'w') as f:
    json.dump(class_names, f)

print(f'\nBest val accuracy: {best_val_acc:.2%}')
print(f'Saved to: {MODEL_DIR}')

In [ ]:
# Evaluation — focus on psoriasis recall (false negatives are the risk)
model.load_state_dict(torch.load(MODEL_DIR / 'best_model.pt', map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        out = model(imgs.to(device))
        all_preds.extend(out.argmax(1).cpu().tolist())
        all_labels.extend(labels.tolist())

correct = sum(p == l for p, l in zip(all_preds, all_labels))
print(f'Overall accuracy: {correct/len(all_labels):.2%}\n')

# psoriasis index
pso_idx = class_names.index('psoriasis')
not_idx = class_names.index('not_psoriasis')

tp = sum(p == pso_idx and l == pso_idx for p, l in zip(all_preds, all_labels))
fp = sum(p == pso_idx and l != pso_idx for p, l in zip(all_preds, all_labels))
fn = sum(p != pso_idx and l == pso_idx for p, l in zip(all_preds, all_labels))
tn = sum(p != pso_idx and l != pso_idx for p, l in zip(all_preds, all_labels))

precision = tp / (tp + fp) if (tp + fp) else 0
recall    = tp / (tp + fn) if (tp + fn) else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

print(f'Psoriasis detection:')
print(f'  Precision: {precision:.2%}  (of predicted psoriasis, how many correct)')
print(f'  Recall:    {recall:.2%}  (of actual psoriasis, how many caught) <- most important')
print(f'  F1:        {f1:.2%}')
print(f'\nConfusion matrix:')
print(f'                    Predicted NOT  Predicted PSO')
print(f'  Actual NOT        {tn:>13}  {fp:>13}')
print(f'  Actual PSO        {fn:>13}  {tp:>13}')
print(f'\nFalse negatives (missed psoriasis): {fn} ({fn/max(tp+fn,1):.1%} of real cases)')

In [ ]:
# Threshold sweep — find optimal cutoff for psoriasis probability
# Run after evaluation to calibrate THRESHOLD in app.py

model.load_state_dict(torch.load(MODEL_DIR / 'best_model.pt', map_location=device))
model.eval()

all_probs, all_labels_sweep = [], []
pso_idx = class_names.index('psoriasis')

with torch.no_grad():
    for imgs, labels in val_loader:
        out = model(imgs.to(device))
        probs = torch.softmax(out, dim=1)[:, pso_idx]
        all_probs.extend(probs.cpu().tolist())
        all_labels_sweep.extend(labels.tolist())

print(f'{"Threshold":>10} {"Precision":>10} {"Recall":>8} {"F1":>6} {"FP":>5} {"FN":>5}')
print('-' * 50)
best_f1, best_thresh = 0, 0.5
for thresh in [t/100 for t in range(30, 80, 5)]:
    tp = sum(p >= thresh and l == pso_idx for p, l in zip(all_probs, all_labels_sweep))
    fp = sum(p >= thresh and l != pso_idx for p, l in zip(all_probs, all_labels_sweep))
    fn = sum(p <  thresh and l == pso_idx for p, l in zip(all_probs, all_labels_sweep))
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall    = tp / (tp + fn) if (tp + fn) else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    marker = ' <-- best F1' if f1 > best_f1 else ''
    if f1 > best_f1:
        best_f1, best_thresh = f1, thresh
    print(f'{thresh:>10.2f} {precision:>10.2%} {recall:>8.2%} {f1:>6.2%} {fp:>5} {fn:>5}{marker}')

print(f'\nBest threshold by F1: {best_thresh}')
print(f'>>> Set THRESHOLD = {best_thresh} in app.py <<<')

## After training
Download from Drive:
- `dermAI_model/best_model.pt`
- `dermAI_model/class_names.json`

Copy to `C:/DEV/Pso/model/` and run `python app.py`.